# 12 — N-Block (NMOS Super-Class AB Input Stage)

**Dataset:** `datasets/n_block/`

Topology: 2 FVF input buffers + NMOS diff pair + NMOS output CM

Nodes: `IBIAS1 IBIAS2 ILCM1 ILCM2 IFVF1 IFVF2 INP INM Min1_D Min2_D OUT_N_1 OUT_N_2 GND`

In [ ]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel


In [ ]:
inp    = gl.Net('inp')
inm    = gl.Net('inm')
ibias1 = gl.Net('ibias1')   # high bias
ibias2 = gl.Net('ibias2')   # low bias
ifvf1  = gl.Net('ifvf1')    # FVF1 Ib node (INP side)
ifvf2  = gl.Net('ifvf2')    # FVF2 Ib node (INM side)
vmid_p = gl.Net('vmid_p')   # FVF1 source/output
vmid_m = gl.Net('vmid_m')   # FVF2 source/output
min1d  = gl.Net('min1d')    # diff pair M+ drain
min2d  = gl.Net('min2d')    # diff pair M- drain
ilcm1  = gl.Net('ilcm1')    # output CM copy 1
ilcm2  = gl.Net('ilcm2')    # output CM copy 2
out_n1 = gl.Net('out_n_1')
out_n2 = gl.Net('out_n_2')

# ── 2× FVF input buffers (INP side + INM side) ───────────────────────
m_fvf1_main = gl.nmos(w=4.0, fingers=1, g=inp,   d=ifvf1,  s=vmid_p)
m_fvf1_fb   = gl.nmos(w=4.0, fingers=1, g=ifvf1, d=vmid_p, s=gl.gnd)
m_fvf2_main = gl.nmos(w=4.0, fingers=1, g=inm,   d=ifvf2,  s=vmid_m)
m_fvf2_fb   = gl.nmos(w=4.0, fingers=1, g=ifvf2, d=vmid_m, s=gl.gnd)

# ── NMOS diff pair (driven by FVF outputs) ───────────────────────────
m_inp = gl.nmos(w=2.25, fingers=1, g=vmid_p, d=min1d, s=ibias1)
m_inm = gl.nmos(w=2.25, fingers=1, g=vmid_m, d=min2d, s=ibias1)

# ── Output current mirrors (NMOS) ────────────────────────────────────
m_cm1_ref  = gl.nmos(w=2.25, g=min1d,  d=min1d,  s=gl.gnd)  # diode ref → OUT_N_1
m_cm1_copy = gl.nmos(w=2.25, g=min1d,  d=ilcm1,  s=gl.gnd)
m_cm2_ref  = gl.nmos(w=2.25, g=min2d,  d=min2d,  s=gl.gnd)  # diode ref → OUT_N_2
m_cm2_copy = gl.nmos(w=2.25, g=min2d,  d=ilcm2,  s=gl.gnd)

# ── Tail bias (from ibias2, simplified as single NMOS ref) ───────────
m_bias = gl.nmos(w=8.3, fingers=1, g=ibias2, d=ibias2, s=gl.gnd)

chip = gl.build(
    m_fvf1_main, m_fvf1_fb, m_fvf2_main, m_fvf2_fb,
    m_inp, m_inm,
    m_cm1_ref, m_cm1_copy, m_cm2_ref, m_cm2_copy,
    m_bias,
    name='n_block'
)
chip.show()
chip.drc()
chip.sim()